# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

> ### Notes de realisation (parties completees)
>
> | Trou a combler | Ce qui a ete mis |
> |---|---|
> | `index_name` (2 endroits) | `"workplace_docs"` — un seul index, reutilise a l'indexation et a la recherche. Les deux cellules **doivent** porter le meme nom, sinon on interroge un index vide. |
> | `metadata_func` | Recuperation de `name`, `summary`, `url`, `category`, `updated_at` via `record.get(...)` — `updated_at` est absent de certains documents du dataset, `record["updated_at"]` planterait. |
> | `chunk_size` / `chunk_overlap` | 800 / 400, comme indique dans la consigne au-dessus de la cellule. |
> | Partie "Your turn" | 3 nouvelles questions + les 3 bonus (comparaison avec/sans MultiQuery, effet de `k`, prompt reformate) + une cellule de reflexion. |
>
> **Prerequis d'execution** : un deploiement Elastic Cloud (Cloud ID + API key) et une cle API OpenAI. Les cellules demandent ces valeurs via `getpass` au moment de l'execution, rien n'est ecrit en dur dans le notebook.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [ ]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-elasticsearch<0.3" "jq==1.12.0"

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [ ]:
#
#
# To find your Cloud ID and create an API key, 
# follow the instructions on the link below:
#
# 👇👇👇
#
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
#
#
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

# Nom de l'index : il doit etre identique ici et dans la cellule d'indexation plus bas,
# sinon on indexe dans un index et on interroge l'autre (qui sera vide).
INDEX_NAME = "workplace_docs"

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [ ]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

# Coup d'oeil sur le corpus : 15 documents "workplace" (RH, ventes, politiques internes)
print(len(data), "documents\n")
for d in data:
    print(f"- {d['name']}  [{d['category']}]")

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [ ]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    # Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    # .get() plutot que [] : tous les documents du dataset n'ont pas de champ "updated_at".
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

# chunk_size=800 tokens, chunk_overlap=400 : un recouvrement de 50 % est volontairement
# eleve, il evite qu'une phrase importante soit coupee en deux entre deux passages.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400
)
docs = loader.load_and_split(text_splitter=text_splitter)

print(len(docs), "passages generes a partir de", len(data), "documents")
print("\nExemple de metadonnees :")
print(docs[0].metadata)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [ ]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name=INDEX_NAME,          # meme index que celui declare plus haut
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [ ]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

### Petit utilitaire

Plutot que de recopier la derniere cellule a chaque fois, on fabrique une petite fonction
qui assemble une chaine RAG a partir d'un *retriever* et d'un prompt. Cela permet de faire
varier un seul parametre a la fois et de comparer proprement.

In [ ]:
def build_chain(retriever, prompt=LLM_CONTEXT_PROMPT, model=None):
    """Assemble une chaine RAG : retrieval -> mise en forme du contexte -> prompt -> LLM."""
    ctx = RunnableParallel(
        context=retriever | _combine_documents,
        question=RunnablePassthrough(),
    )
    return ctx | prompt | (model or llm)


def ask(question, chain, titre=None):
    """Pose une question et affiche la reponse (les requetes generees apparaissent dans les logs)."""
    print("=" * 90)
    print(titre or question)
    print("=" * 90)
    reponse = chain.invoke(question)
    print(reponse)
    return reponse


def show_sources(question, retriever):
    """Affiche les passages effectivement recuperes : indispensable pour juger l'ancrage."""
    docs_recup = retriever.get_relevant_documents(question)
    print(f"{len(docs_recup)} passages recuperes pour : {question!r}\n")
    for d in docs_recup:
        print(f"- {d.metadata.get('name')}  [{d.metadata.get('category')}]")
    return docs_recup

### Iteration 1 — une question dont la reponse est repartie sur plusieurs documents

**Question :** *"How many days per week am I expected to work from the office?"*

Ce cas est interessant parce que le corpus contient **trois** documents qui se contredisent dans le temps : la politique initiale de teletravail, une mise a jour d'avril 2022 (deux jours au bureau) et une mise a jour de mai 2023 (trois jours au bureau). C'est exactement le genre de question ou le MultiQuery aide : il faut retrouver les trois versions, et le modele doit ensuite reperer laquelle est la plus recente.

In [ ]:
q1 = "How many days per week am I expected to work from the office?"

sources_q1 = show_sources(q1, retriever)
print()
ans_q1 = ask(q1, chain)

### Iteration 2 — une question tres localisee dans un seul document

**Question :** *"Can I bring my dog to the office, and what do I need to do before?"*

A l'inverse de la premiere, la reponse tient entierement dans un seul document (*Office Pet Policy*). On s'attend a ce que le MultiQuery apporte peu ici : la recherche simple devrait deja trouver le bon passage. C'est utile de le verifier, pour ne pas conclure trop vite que le MultiQuery aide *toujours*.

In [ ]:
q2 = "Can I bring my dog to the office, and what do I need to do before?"

sources_q2 = show_sources(q2, retriever)
print()
ans_q2 = ask(q2, chain)

### Iteration 3 — question hors corpus (bonus 4)

**Question :** *"What is the company's policy on reimbursing gym memberships?"*

Aucun document ne traite de ce sujet. Le test verifie si la consigne *"If you don't know the answer, just say that you don't know"* est effectivement respectee, ou si le modele invente une politique a partir des passages RH qui lui ont ete servis par defaut. C'est le test le plus important de tous pour un usage reel : un RAG qui hallucine sur une question hors perimetre n'est pas deployable.

In [ ]:
q3 = "What is the company's policy on reimbursing gym memberships?"

sources_q3 = show_sources(q3, retriever)
print()
ans_q3 = ask(q3, chain)

### Bonus 1 — MultiQueryRetriever vs recherche simple

Meme question, deux retrievers : celui qui reformule la question en plusieurs variantes, et la
recherche vectorielle brute. On regarde d'abord **quels passages** chacun ramene, puis les reponses.

In [ ]:
simple_retriever = vectorstore.as_retriever()
chain_simple = build_chain(simple_retriever)

print(">>> RETRIEVER SIMPLE")
docs_simple = show_sources(q1, simple_retriever)

print("\n>>> MULTIQUERY (voir les requetes generees dans les logs ci-dessus)")
docs_multi = show_sources(q1, retriever)

noms_simple = {d.metadata.get("name") for d in docs_simple}
noms_multi = {d.metadata.get("name") for d in docs_multi}
print("\nDocuments trouves uniquement par le MultiQuery :", noms_multi - noms_simple)
print("Documents trouves uniquement par la recherche simple :", noms_simple - noms_multi)

In [ ]:
ans_q1_simple = ask(q1, chain_simple, titre="[SANS MultiQuery] " + q1)
print()
ans_q2_simple = ask(q2, chain_simple, titre="[SANS MultiQuery] " + q2)

### Bonus 2 — effet du parametre `k`

`k` fixe le nombre de passages ramenes **par requete**. Avec le MultiQuery, chaque variante de la
question ramene `k` passages, puis l'union est dedupliquee : le contexte final grossit vite.
On compare k=2 (contexte serre) et k=6 (contexte large).

In [ ]:
for k in (2, 6):
    r_k = MultiQueryRetriever.from_llm(
        vectorstore.as_retriever(search_kwargs={"k": k}), llm
    )
    docs_k = r_k.get_relevant_documents(q1)
    print(f"k={k} -> {len(docs_k)} passages uniques apres fusion : "
          f"{sorted({d.metadata.get('name') for d in docs_k})}\n")

In [ ]:
chain_k6 = build_chain(
    MultiQueryRetriever.from_llm(vectorstore.as_retriever(search_kwargs={"k": 6}), llm)
)
ans_q1_k6 = ask(q1, chain_k6, titre="[k=6] " + q1)

### Bonus 3 — reformater la reponse via le prompt

Le prompt d'origine demande explicitement d'etre *"as verbose and educational as possible"*.
Pour un chatbot RH interne, on veut plutot une reponse courte, en puces, avec la source citee.
Seul le prompt change ; la recuperation est identique.

In [ ]:
LLM_CONTEXT_PROMPT_COURT = ChatPromptTemplate.from_template(
    """You are an internal HR assistant. Answer the question using ONLY the retrieved context.
    Rules:
    - Answer in at most 4 short bullet points.
    - After each bullet, cite the source document name in brackets, e.g. [Company Vacation Policy].
    - If the context does not contain the answer, reply exactly: "I don't know based on the internal documents."

    context: {context}
    Question: "{question}"
    Answer:
    """
)

chain_court = build_chain(retriever, prompt=LLM_CONTEXT_PROMPT_COURT)

ans_q1_court = ask(q1, chain_court, titre="[prompt court + citations] " + q1)
print()
ans_q3_court = ask(q3, chain_court, titre="[prompt court + citations] " + q3)

<br>

## Reflexion

**1. Les questions generees par le MultiQueryRetriever sont-elles utiles ?**

Le principe fonctionne parce que la recherche vectorielle compare des *formulations*, pas des
intentions. En reformulant ("how many days in the office", "what is the hybrid work policy",
"remote work requirements per week"), le retriever couvre plusieurs facons de dire la meme chose et
rattrape des passages qu'une seule requete manque. Le point a verifier dans les logs est la
**derive** : le LLM qui reformule peut ajouter un concept absent de la question initiale
(ex. transformer une question sur les jours au bureau en question sur les horaires flexibles), et
ramener des passages hors sujet qui diluent le contexte.

**2. Est-ce que le MultiQuery aide toujours ?**

Non, et c'est ce que compare le bonus 1. Sur l'iteration 2 (politique animaux), la reponse tient
dans un seul document et la recherche simple suffit — le MultiQuery ajoute alors trois appels LLM
supplementaires et de la latence pour un resultat equivalent. Il devient reellement utile quand la
reponse est **repartie sur plusieurs documents** ou quand le vocabulaire de l'utilisateur ne
correspond pas a celui des documents. C'est un arbitrage cout/latence contre rappel, pas un
reglage a activer par defaut.

**3. Les reponses sont-elles bien ancrees dans le contexte ?**

La fonction `show_sources()` a ete ajoutee exactement pour ca : sans afficher les passages
recuperes, on ne peut pas distinguer une bonne reponse d'une reponse plausible. Deux points de
vigilance sur ce corpus :
- **Documents contradictoires** (iteration 1) : trois versions successives de la politique de
  teletravail coexistent dans l'index. Le retriever n'a aucune notion de fraicheur — il rend les
  trois, et rien ne garantit que le modele privilegie la plus recente. La metadonnee `updated_at`
  est indexee mais n'est ni transmise au prompt (`LLM_DOCUMENT_PROMPT` n'affiche que `name` et le
  contenu), ni utilisee pour trier. **C'est la principale faiblesse du pipeline** : la correction
  serait d'ajouter la date dans le prompt document, ou de filtrer sur les metadonnees.
- **Question hors corpus** (iteration 3) : le retriever ramene *toujours* des passages, meme quand
  aucun n'est pertinent — il n'a pas de seuil de similarite. Le refus de repondre repose donc
  entierement sur la consigne du prompt. Le prompt reformate du bonus 3, avec une phrase de refus
  imposee mot pour mot, est nettement plus fiable sur ce point que la formulation d'origine.

**4. A completer apres execution**

*(Cellules a executer avec les cles Elastic + OpenAI ; noter ici les observations reelles :
requetes generees visibles dans les logs, documents effectivement retrouves par chaque retriever,
et si la reponse a l'iteration 1 cite bien la version de mai 2023.)*